# LM Alignment: Simple Explanations

Full alignment pipeline for **Qwen/Qwen3-4B-Instruct-2507** via QLoRA.

| Task | Method | Train data | Metric |
|------|--------|-----------|--------|
| 1 | SFT | kid_adult | P_simple |
| 2 | DPO style | kid_adult | P_simple |
| 3 | Reward Model | good_bad | RM accuracy |
| 4 | DPO quality | good_bad | gold_rm score |
| 5 | SimPO | good_bad | gold_rm score |

**Requirements:** T4 GPU (15 GB VRAM), Google Drive with the `ml-olympiad-red-task` folder uploaded.

**Setup before first run:**
1. Upload the entire `ml-olympiad-red-task/` folder to Google Drive root.
2. (Optional) Add `HF_TOKEN` to Colab secrets for Hub access.


In [ ]:
# ── 0. Install ──────────────────────────────────────────────
!pip install -q transformers==4.51.3 trl==0.12.2 peft==0.14.0 \
    bitsandbytes==0.44.1 accelerate==1.2.1 datasets==3.2.0 \
    safetensors scikit-learn scipy
print('✅ All packages installed')

In [ ]:
# ── 1. Global config & reproducibility ──────────────────────
import os, gc, re, json, random, pickle
import numpy as np
import torch

SEED = 42
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

METRICS = {}  # collects all final metric values

In [ ]:
# ── 2. Mount Google Drive & copy data locally ────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil

DRIVE_BASE   = "/content/drive/MyDrive/ml-olympiad-red-task"
LOCAL_BASE   = "/content/task"
DATA_DIR     = f"{LOCAL_BASE}/data"
METRICS_DIR  = f"{LOCAL_BASE}/metrics"
GOLD_RM_PATH = f"{METRICS_DIR}/gold_rm"

os.makedirs(LOCAL_BASE, exist_ok=True)

for subdir in ["data", "metrics"]:
    dst = os.path.join(LOCAL_BASE, subdir)
    src = os.path.join(DRIVE_BASE, subdir)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f"Copied {subdir}/")
    else:
        print(f"Already exists: {subdir}/")

print("Data files :", os.listdir(DATA_DIR))
print("Metric files:", os.listdir(METRICS_DIR))

In [ ]:
# ── 3. Load raw datasets ─────────────────────────────────────
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

kid_adult_raw    = load_jsonl(f"{DATA_DIR}/kid_adult.jsonl")
good_bad_raw     = load_jsonl(f"{DATA_DIR}/good_bad.jsonl")
test_style_raw   = load_jsonl(f"{DATA_DIR}/public_test_style.jsonl")
test_quality_raw = load_jsonl(f"{DATA_DIR}/public_test_quality.jsonl")

print(f"kid_adult    : {len(kid_adult_raw):>5}  keys={list(kid_adult_raw[0].keys())}")
print(f"good_bad     : {len(good_bad_raw):>5}  keys={list(good_bad_raw[0].keys())}")
print(f"test_style   : {len(test_style_raw):>5}")
print(f"test_quality : {len(test_quality_raw):>5}")

TEST_STYLE_PROMPTS   = [d["prompt"] for d in test_style_raw]
TEST_QUALITY_PROMPTS = [d["prompt"] for d in test_quality_raw]
TEST_QUALITY_CHOSEN  = [d["chosen"] for d in test_quality_raw]
TEST_QUALITY_REJECTED= [d["rejected"] for d in test_quality_raw]

In [ ]:
# ── 4. QLoRA / LoRA configs ───────────────────────────────────
from transformers import BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Shared LoRA for causal LM tasks (SFT, DPO, SimPO)
lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# LoRA for Sequence Classification (Task 3 RM)
lora_cfg_cls = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

def free_mem(*objs):
    for o in objs:
        try: del o
        except: pass
    gc.collect()
    torch.cuda.empty_cache()

print("Configs ready.")

In [ ]:
# ── 5. Style classifier & P_simple metric ───────────────────
from scipy.sparse import hstack as sp_hstack

with open(f"{METRICS_DIR}/style_clf.pkl", "rb") as f:
    _clf_bundle = pickle.load(f)

# style_clf.pkl is a dict: {'clf': LogisticRegression, 'vecs': (TfidfVectorizer, TfidfVectorizer)}
print("Bundle type:", type(_clf_bundle))
if isinstance(_clf_bundle, dict):
    _style_lr   = _clf_bundle['clf']
    _style_vecs = _clf_bundle['vecs']   # tuple of two TfidfVectorizers
    print("clf:", type(_style_lr).__name__)
    print("vecs:", [type(v).__name__ for v in _style_vecs])
    def _style_predict_proba(texts):
        X = sp_hstack([v.transform(texts) for v in _style_vecs])
        return _style_lr.predict_proba(X)
else:
    # fallback: plain sklearn pipeline
    _style_predict_proba = _clf_bundle.predict_proba

# Determine which column = simple
_tests = [
    "Это простой ответ для детей с понятными аналогиями.",
    "Данный феномен обусловлен многоуровневым взаимодействием сложных когнитивных процессов."
]
_p = _style_predict_proba(_tests)
SIMPLE_IDX = 1 if _p[0][1] > _p[0][0] else 0
print(f"Simple class index : {SIMPLE_IDX}")
print(f"  simple text  P_simple={_p[0][SIMPLE_IDX]:.3f}")
print(f"  complex text P_simple={_p[1][SIMPLE_IDX]:.3f}")

def compute_p_simple(texts):
    """Mean P(simple) over a list of texts."""
    safe = [t if t and t.strip() else "нет ответа" for t in texts]
    proba = _style_predict_proba(safe)
    return float(np.mean(proba[:, SIMPLE_IDX]))

In [ ]:
# ── 6. Generation helper (greedy, seed=42) ───────────────────
from transformers import AutoTokenizer

def apply_tmpl(tokenizer, messages, gen_prompt=True):
    """apply_chat_template with enable_thinking=False if supported."""
    kwargs = dict(tokenize=False, add_generation_prompt=gen_prompt)
    try:
        return tokenizer.apply_chat_template(messages, **kwargs, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)

def strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def generate_responses(model, tokenizer, prompts, max_new_tokens=256):
    """Greedy generation, deterministic (do_sample=False)."""
    torch.manual_seed(SEED)
    model.eval()
    responses = []
    for prompt in prompts:
        msgs = [{"role": "user", "content": prompt}]
        text = apply_tmpl(tokenizer, msgs, gen_prompt=True)
        inputs = tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=768
        ).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        new_ids = out[0][inputs["input_ids"].shape[1]:]
        resp = tokenizer.decode(new_ids, skip_special_tokens=True)
        responses.append(strip_think(resp))
    return responses

print("Generation helper ready.")

In [ ]:
# ── 7. Gold RM loader & scorer ───────────────────────────────
# Safety install in case session was restarted after cell 0
import importlib, subprocess, sys
for _pkg in ['trl', 'peft', 'safetensors']:
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])

from transformers import AutoModelForCausalLM
from peft import PeftModel
from safetensors.torch import load_file as load_sf
from trl import AutoModelForCausalLMWithValueHead

def load_gold_rm():
    """Load gold reward model (base + LoRA adapter + value head)."""
    tok = AutoTokenizer.from_pretrained(GOLD_RM_PATH, trust_remote_code=True)
    tok.pad_token = tok.pad_token or tok.eos_token
    tok.padding_side = "right"

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base, GOLD_RM_PATH, is_trainable=False)
    peft_model.eval()

    rm = AutoModelForCausalLMWithValueHead(peft_model)
    vh = load_sf(f"{GOLD_RM_PATH}/value_head.safetensors")
    vh = {k: v.to(torch.bfloat16) for k, v in vh.items()}
    rm.v_head.load_state_dict(vh, strict=False)
    rm.eval()
    return rm, tok

@torch.no_grad()
def score_with_rm(rm, tok, prompts, responses):
    """Score each (prompt, response) pair; returns list of floats."""
    scores = []
    rm.eval()
    for p, r in zip(prompts, responses):
        msgs = [
            {"role": "user",      "content": p},
            {"role": "assistant", "content": r},
        ]
        text = apply_tmpl(tok, msgs, gen_prompt=False)
        inp  = tok(text, return_tensors="pt",
                   truncation=True, max_length=1024).to(DEVICE)
        _, _, values = rm(**inp)
        if values.dim() == 3:
            values = values.squeeze(-1)
        last = inp["attention_mask"].sum() - 1
        scores.append(values[0, last].item())
    return scores

print("Gold RM helpers ready.")

---
## Task 1 — SFT on `kid_adult`
**Goal:** teach the model to respond in simple style by fine-tuning on (prompt → kid) pairs.  
**Metric:** P_simple on `public_test_style`


In [ ]:
# ── Task 1 · Prepare SFT dataset ────────────────────────────
from datasets import Dataset

# Load tokenizer once to pre-format text
_tok_tmp = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
_tok_tmp.pad_token = _tok_tmp.pad_token or _tok_tmp.eos_token

def fmt_sft(ex):
    msgs = [
        {"role": "user",      "content": ex["prompt"]},
        {"role": "assistant", "content": ex["kid"]},
    ]
    return {"text": apply_tmpl(_tok_tmp, msgs, gen_prompt=False)}

sft_ds = Dataset.from_list(kid_adult_raw).map(fmt_sft, remove_columns=list(kid_adult_raw[0].keys()))
del _tok_tmp
print(f"SFT dataset: {len(sft_ds)} examples")
print("Sample (first 200 chars):", sft_ds[0]["text"][:200])

In [ ]:
# ── Task 1 · Train SFT ───────────────────────────────────────
from transformers import AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

SFT_OUT = "/content/sft_adapter"

torch.manual_seed(SEED)
model_sft = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model_sft = prepare_model_for_kbit_training(model_sft)

tok_sft = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_sft.pad_token = tok_sft.pad_token or tok_sft.eos_token
tok_sft.padding_side = "right"

sft_cfg = SFTConfig(
    output_dir=SFT_OUT,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=512,
    dataset_text_field="text",
    save_strategy="no",
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model_sft,
    args=sft_cfg,
    train_dataset=sft_ds,
    processing_class=tok_sft,
    peft_config=lora_cfg,
)
sft_trainer.train()
sft_trainer.save_model(SFT_OUT)
tok_sft.save_pretrained(SFT_OUT)
print(f"SFT saved → {SFT_OUT}")

In [ ]:
# ── Task 1 · Evaluate → METRIC 1 ────────────────────────────
free_mem(model_sft, sft_trainer)

# Load adapter in bfloat16 for inference
eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
eval_model = PeftModel.from_pretrained(eval_base, SFT_OUT, is_trainable=False)
eval_tok   = AutoTokenizer.from_pretrained(SFT_OUT, trust_remote_code=True)
eval_tok.pad_token = eval_tok.pad_token or eval_tok.eos_token

sft_resps  = generate_responses(eval_model, eval_tok, TEST_STYLE_PROMPTS)
metric_1   = compute_p_simple(sft_resps)
METRICS["task1_sft_p_simple"] = metric_1

print(f"\n{'='*55}")
print(f"METRIC 1  SFT P_simple = {metric_1:.4f}")
print(f"{'='*55}")
for i in range(2):
    print(f"\n  Prompt  : {TEST_STYLE_PROMPTS[i][:80]}...")
    print(f"  Response: {sft_resps[i][:120]}...")

free_mem(eval_model, eval_base)

---
## Task 2 — DPO (style) on `kid_adult`
**Goal:** preference optimisation — push the model to prefer the simple (kid) response over the complex (adult) one.  
**Metric:** P_simple on `public_test_style`


In [ ]:
# ── Task 2 · Prepare DPO-style dataset ──────────────────────
from trl import DPOTrainer, DPOConfig

def fmt_dpo_style(ex):
    return {
        "prompt":   [{"role": "user",      "content": ex["prompt"]}],
        "chosen":   [{"role": "assistant", "content": ex["kid"]}],
        "rejected": [{"role": "assistant", "content": ex["adult"]}],
    }

dpo_style_ds = Dataset.from_list(kid_adult_raw).map(
    fmt_dpo_style, remove_columns=list(kid_adult_raw[0].keys())
)
print(f"DPO style dataset: {len(dpo_style_ds)} examples")

In [ ]:
# ── Task 2 · Train DPO style ─────────────────────────────────
DPO_STYLE_OUT = "/content/dpo_style_adapter"

torch.manual_seed(SEED)
model_dpo_s = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model_dpo_s = prepare_model_for_kbit_training(model_dpo_s)

tok_dpo_s = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_dpo_s.pad_token = tok_dpo_s.pad_token or tok_dpo_s.eos_token
tok_dpo_s.padding_side = "left"   # DPO pads prompts from the left

dpo_s_cfg = DPOConfig(
    output_dir=DPO_STYLE_OUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    save_strategy="no",
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

dpo_s_trainer = DPOTrainer(
    model=model_dpo_s,
    ref_model=None,
    args=dpo_s_cfg,
    train_dataset=dpo_style_ds,
    processing_class=tok_dpo_s,
    peft_config=lora_cfg,
)
dpo_s_trainer.train()
dpo_s_trainer.save_model(DPO_STYLE_OUT)
tok_dpo_s.save_pretrained(DPO_STYLE_OUT)
print(f"DPO style saved → {DPO_STYLE_OUT}")

In [ ]:
# ── Task 2 · Evaluate → METRIC 2 ────────────────────────────
free_mem(model_dpo_s, dpo_s_trainer)

eval_base2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
eval_m2    = PeftModel.from_pretrained(eval_base2, DPO_STYLE_OUT, is_trainable=False)
eval_tok2  = AutoTokenizer.from_pretrained(DPO_STYLE_OUT, trust_remote_code=True)
eval_tok2.pad_token = eval_tok2.pad_token or eval_tok2.eos_token

dpo_s_resps = generate_responses(eval_m2, eval_tok2, TEST_STYLE_PROMPTS)
metric_2    = compute_p_simple(dpo_s_resps)
METRICS["task2_dpo_style_p_simple"] = metric_2

print(f"\n{'='*55}")
print(f"METRIC 2  DPO-style P_simple = {metric_2:.4f}")
print(f"{'='*55}")

free_mem(eval_m2, eval_base2)

---
## Task 3 — Reward Model on `good_bad`
**Goal:** train a model to distinguish a *good* simple explanation from a *bad* one (both are simple — quality axis).  
**Metric:** accuracy on `public_test_quality` — fraction where our RM scores `chosen` > `rejected`


In [ ]:
# ── Task 3 · Prepare RM dataset ──────────────────────────────
from trl import RewardTrainer, RewardConfig

def fmt_rm(ex):
    return {
        "chosen":   [{"role": "user", "content": ex["instruction"]},
                     {"role": "assistant", "content": ex["chosen"]}],
        "rejected": [{"role": "user", "content": ex["instruction"]},
                     {"role": "assistant", "content": ex["rejected"]}],
    }

rm_ds = Dataset.from_list(good_bad_raw).map(fmt_rm, remove_columns=list(good_bad_raw[0].keys()))
print(f"RM dataset: {len(rm_ds)} examples")

In [ ]:
# ── Task 3 · Train Reward Model ──────────────────────────────
from transformers import AutoModelForSequenceClassification

RM_OUT = "/content/reward_model"

torch.manual_seed(SEED)
model_rm = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model_rm = prepare_model_for_kbit_training(model_rm)

tok_rm = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_rm.pad_token = tok_rm.pad_token or tok_rm.eos_token
tok_rm.padding_side = "right"

rm_cfg = RewardConfig(
    output_dir=RM_OUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=512,
    save_strategy="no",
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
)

rm_trainer = RewardTrainer(
    model=model_rm,
    args=rm_cfg,
    train_dataset=rm_ds,
    processing_class=tok_rm,
    peft_config=lora_cfg_cls,
)
rm_trainer.train()
rm_trainer.save_model(RM_OUT)
tok_rm.save_pretrained(RM_OUT)
print(f"Reward model saved → {RM_OUT}")

In [ ]:
# ── Task 3 · Evaluate → METRIC 3 ────────────────────────────
free_mem(model_rm, rm_trainer)

# Load trained RM in bfloat16
trained_rm_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1,
    torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
trained_rm = PeftModel.from_pretrained(trained_rm_base, RM_OUT, is_trainable=False)
trained_rm.eval()

trained_rm_tok = AutoTokenizer.from_pretrained(RM_OUT, trust_remote_code=True)
trained_rm_tok.pad_token = trained_rm_tok.pad_token or trained_rm_tok.eos_token

@torch.no_grad()
def score_cls_rm(rm, tok, prompts, responses):
    scores = []
    for p, r in zip(prompts, responses):
        msgs = [
            {"role": "user",      "content": p},
            {"role": "assistant", "content": r},
        ]
        text = apply_tmpl(tok, msgs, gen_prompt=False)
        inp  = tok(text, return_tensors="pt",
                   truncation=True, max_length=512).to(DEVICE)
        out  = rm(**inp)
        scores.append(out.logits[0, 0].item())
    return scores

chosen_sc   = score_cls_rm(trained_rm, trained_rm_tok,
                            TEST_QUALITY_PROMPTS, TEST_QUALITY_CHOSEN)
rejected_sc = score_cls_rm(trained_rm, trained_rm_tok,
                            TEST_QUALITY_PROMPTS, TEST_QUALITY_REJECTED)

correct  = sum(c > r for c, r in zip(chosen_sc, rejected_sc))
metric_3 = correct / len(TEST_QUALITY_PROMPTS)
METRICS["task3_rm_accuracy"] = metric_3

print(f"\n{'='*55}")
print(f"METRIC 3  RM accuracy = {metric_3:.4f}  ({correct}/{len(TEST_QUALITY_PROMPTS)})")
print(f"  Mean chosen   score : {np.mean(chosen_sc):.4f}")
print(f"  Mean rejected score : {np.mean(rejected_sc):.4f}")
print(f"{'='*55}")

free_mem(trained_rm, trained_rm_base)

---
## Task 4 — DPO (quality) on `good_bad`
**Goal:** preference optimisation on quality axis — both responses are simple, but chosen is more accurate/helpful.  
**Metric:** mean reward from `gold_rm` on generated responses for `public_test_quality` prompts


In [ ]:
# ── Task 4 · Prepare DPO-quality dataset ─────────────────────
def fmt_dpo_quality(ex):
    return {
        "prompt":   [{"role": "user",      "content": ex["instruction"]}],
        "chosen":   [{"role": "assistant", "content": ex["chosen"]}],
        "rejected": [{"role": "assistant", "content": ex["rejected"]}],
    }

dpo_q_ds = Dataset.from_list(good_bad_raw).map(
    fmt_dpo_quality, remove_columns=list(good_bad_raw[0].keys())
)
print(f"DPO quality dataset: {len(dpo_q_ds)} examples")

In [ ]:
# ── Task 4 · Train DPO quality ───────────────────────────────
DPO_Q_OUT = "/content/dpo_quality_adapter"

torch.manual_seed(SEED)
model_dpo_q = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model_dpo_q = prepare_model_for_kbit_training(model_dpo_q)

tok_dpo_q = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_dpo_q.pad_token = tok_dpo_q.pad_token or tok_dpo_q.eos_token
tok_dpo_q.padding_side = "left"

dpo_q_cfg = DPOConfig(
    output_dir=DPO_Q_OUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    save_strategy="no",
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

dpo_q_trainer = DPOTrainer(
    model=model_dpo_q,
    ref_model=None,
    args=dpo_q_cfg,
    train_dataset=dpo_q_ds,
    processing_class=tok_dpo_q,
    peft_config=lora_cfg,
)
dpo_q_trainer.train()
dpo_q_trainer.save_model(DPO_Q_OUT)
tok_dpo_q.save_pretrained(DPO_Q_OUT)
print(f"DPO quality saved → {DPO_Q_OUT}")

In [ ]:
# ── Task 4 · Generate responses ──────────────────────────────
free_mem(model_dpo_q, dpo_q_trainer)

eval_base4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
eval_m4   = PeftModel.from_pretrained(eval_base4, DPO_Q_OUT, is_trainable=False)
eval_tok4 = AutoTokenizer.from_pretrained(DPO_Q_OUT, trust_remote_code=True)
eval_tok4.pad_token = eval_tok4.pad_token or eval_tok4.eos_token

dpo_q_resps = generate_responses(eval_m4, eval_tok4, TEST_QUALITY_PROMPTS)
free_mem(eval_m4, eval_base4)

In [ ]:
# ── Task 4 · Score with gold_rm → METRIC 4 ──────────────────
gold_rm, gold_rm_tok = load_gold_rm()

dpo_q_scores = score_with_rm(gold_rm, gold_rm_tok, TEST_QUALITY_PROMPTS, dpo_q_resps)
ref_scores   = score_with_rm(gold_rm, gold_rm_tok, TEST_QUALITY_PROMPTS, TEST_QUALITY_REJECTED)

metric_4   = float(np.mean(dpo_q_scores))
win_rate_4 = float(np.mean([s > r for s, r in zip(dpo_q_scores, ref_scores)]))
METRICS["task4_dpo_quality_mean_reward"] = metric_4
METRICS["task4_dpo_quality_win_rate"]    = win_rate_4

print(f"\n{'='*55}")
print(f"METRIC 4  DPO-quality mean gold_rm reward = {metric_4:.4f}")
print(f"          DPO-quality win rate vs rejected = {win_rate_4:.4f}")
print(f"{'='*55}")

free_mem(gold_rm)

---
## Task 5 — SimPO on `good_bad`
**Goal:** reference-free preference optimisation (SimPO) on the quality axis.  
SimPO does not require a reference model — the implicit reference is the length-normalised log-likelihood of the model itself.  
**Metric:** mean reward from `gold_rm` on generated responses for `public_test_quality` prompts


In [ ]:
# ── Task 5 · Train SimPO ─────────────────────────────────────
from trl import CPOTrainer, CPOConfig

SIMPO_OUT = "/content/simpo_adapter"

torch.manual_seed(SEED)
model_simpo = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model_simpo = prepare_model_for_kbit_training(model_simpo)

tok_simpo = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok_simpo.pad_token = tok_simpo.pad_token or tok_simpo.eos_token
tok_simpo.padding_side = "left"

simpo_cfg = CPOConfig(
    output_dir=SIMPO_OUT,
    loss_type="simpo",         # SimPO loss (reference-free)
    beta=2.0,                  # temperature scaling
    simpo_gamma=0.5,           # reward margin
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=512,
    max_prompt_length=256,
    save_strategy="no",
    logging_steps=50,
    bf16=True,
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# Dataset reuses dpo_q_ds — same prompt/chosen/rejected format
simpo_trainer = CPOTrainer(
    model=model_simpo,
    args=simpo_cfg,
    train_dataset=dpo_q_ds,
    processing_class=tok_simpo,
    peft_config=lora_cfg,
)
simpo_trainer.train()
simpo_trainer.save_model(SIMPO_OUT)
tok_simpo.save_pretrained(SIMPO_OUT)
print(f"SimPO saved → {SIMPO_OUT}")

In [ ]:
# ── Task 5 · Generate responses ──────────────────────────────
free_mem(model_simpo, simpo_trainer)

eval_base5 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
eval_m5   = PeftModel.from_pretrained(eval_base5, SIMPO_OUT, is_trainable=False)
eval_tok5 = AutoTokenizer.from_pretrained(SIMPO_OUT, trust_remote_code=True)
eval_tok5.pad_token = eval_tok5.pad_token or eval_tok5.eos_token

simpo_resps = generate_responses(eval_m5, eval_tok5, TEST_QUALITY_PROMPTS)
free_mem(eval_m5, eval_base5)

In [ ]:
# ── Task 5 · Score with gold_rm → METRIC 5 ──────────────────
gold_rm, gold_rm_tok = load_gold_rm()

simpo_scores = score_with_rm(gold_rm, gold_rm_tok, TEST_QUALITY_PROMPTS, simpo_resps)

metric_5   = float(np.mean(simpo_scores))
win_rate_5 = float(np.mean([s > r for s, r in zip(simpo_scores, ref_scores)]))
METRICS["task5_simpo_mean_reward"] = metric_5
METRICS["task5_simpo_win_rate"]    = win_rate_5

print(f"\n{'='*55}")
print(f"METRIC 5  SimPO mean gold_rm reward = {metric_5:.4f}")
print(f"          SimPO win rate vs rejected = {win_rate_5:.4f}")
print(f"{'='*55}")

free_mem(gold_rm)

---
## Summary of all metrics


In [ ]:
# ── Final summary ────────────────────────────────────────────
print()
print("╔" + "═"*55 + "╗")
print("║  ALIGNMENT METRICS  (seed=42, greedy, do_sample=False) ║")
print("╠" + "═"*55 + "╣")
print(f"║  Task 1  SFT P_simple              : {METRICS.get('task1_sft_p_simple',float('nan')):.4f}          ║")
print(f"║  Task 2  DPO-style P_simple        : {METRICS.get('task2_dpo_style_p_simple',float('nan')):.4f}          ║")
print(f"║  Task 3  RM accuracy               : {METRICS.get('task3_rm_accuracy',float('nan')):.4f}          ║")
print(f"║  Task 4  DPO-quality mean reward   : {METRICS.get('task4_dpo_quality_mean_reward',float('nan')):.4f}          ║")
print(f"║  Task 4  DPO-quality win rate      : {METRICS.get('task4_dpo_quality_win_rate',float('nan')):.4f}          ║")
print(f"║  Task 5  SimPO mean reward         : {METRICS.get('task5_simpo_mean_reward',float('nan')):.4f}          ║")
print(f"║  Task 5  SimPO win rate            : {METRICS.get('task5_simpo_win_rate',float('nan')):.4f}          ║")
print("╚" + "═"*55 + "╝")
print()
print("All values saved in METRICS dict:", METRICS)